In [ ]:
### Data Preprocessing

## 1. Split

# train / test split
train_ratio = 0.8
train_data_len = math.ceil( len(df) * train_ratio )

train_data = df[ : train_data_len ][ ['Open'] ]
test_data = df[ train_data_len : ][ ['Open'] ]


## 2. Scaling : fit_transform() for train data, transform() for test data

import sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler( feature_range=(0, 1) )
train_scaled = scaler.fit_transform( train_data.values )
test_scaled = scaler.transform( test_data.values )


## 3. Convert into tensors

def convert_data_into_tensors( data_seq ):
    features, labels = [], []

    for i in range( len(data_seq) - sequence_length ):              # sequence_length = sliding window
        features.append( data_seq[ i : i + sequence_length ] )
        labels.append( data_seq[ i + sequence_length : 0 ] )        # 바로 다음 값 예측
    
    features, labels = np.array( features ), np.array( labels )

    features = torch.tensor( features, dtype=torch.float32 )
    labels = torch.tensor( labels, dtype=torch.flaot32 )

    return features, labels

X_train, y_train = convert_data_into_tensors( train_scaled )
X_test, y_test = convert_data_into_tensors( test_scaled )


## 4. Data Loader

def to_loader( x, y, batch_size=32, shuffle ) :
    dataset = torch.utils.data.TensorDataset( x, y )
    return torch.utils.data.DataLoader( dataset, batch_size, shuffle )

train_loader = to_loader( X_train, y_train, batch_size, shuffle=True )
test_loader = to_loader( X_test, y_test, batch_size, shuffle=False )

In [ ]:
### LSTM 모델 구성 및 Train / 평가 함수

## 1. LSTM 모델 class

input_size = X_train.shape[ -1 ]
num_layers = 2
hidden_size = 64

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModl, self).__init__()

        # (lstm) : LSTM( 1, 64, num_layers=2, batch_first=True )
        self.lstm = nn.LSTM( input_size, hidden_size, num_layers, batch_first=True )

        # (linear) : Linear( hidden_size, 1 )
        self.linear = nn.Linear( hidden_size, 1 )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out)

model = LSTMModel( input_size, hidden_size, num_layers ).to( device )


## 2. Training LSTM 모델

def train( model, train_loader, test_loader ) :
    train_hist, test_hist = [], []
    num_epochs = 10

    loss_fn = nn.MSELoss( reduction="mean" )
    optimizer = torch.optim.Adam( model.parameters(), lr=1e-3 )

    for epoch in range(num_epochs):
        total_train_loss = 0.0
        total_test_loss = 0.0

        # train
        model.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()

            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            pred = model( batch_x )[ :, -1, 0 ]     # 마지막 1개 결과 예측
            loss = loss_fn( pred, batch_y )
            
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_loss = total_train_loss / len( train_loader )
        train_hist.append( avg_loss )

        # evalue per each epoch
        model.eval()
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            test_pred = model( batch_x )[ :, -1, 0 ]    # 마지막 1개 결과(예측)
            test_loss = loss_fn( pred, batch_y )

            total_test_loss += test_loss.item()

        avg_test_loss = total_test_loss / len( test_loader )
        test_hist.append( avg_test_loss )

train( model, train_loader, test_loader )


## 3. 평가(Test) 함수

def test( model, X_test, y_test ):
    model.eval()
    
    with torch.no_grad():
        X_test = X_test.to(device)              # [N, sequence_length, 1]
        y_pred = model( X_test )                # [N, sequence_length, 1]
        test_predictions = y_pred[:, -1, 0]     # [N, 1, 1] : 마지막 sequnce 의 결과 예측 값

    test_predictions = test_predictions.cpu().numpy()
    y_test = y_test.cpu().numpy()

    rmse = np.sqrt( mean_squared_error( y_test, test_predictions ) )
    mape = mean_absolute_precentage_error( y_test, test_predictions )

    return rmse, mape

In [ ]:
### CNN 모델 구성 (학습 및 평가 함수는 동일)
    - kernel_size = 2 이므로 앞 2개 값을 읽어서 뒤 1개 값을 예측하는 모델이 됨
    - 그래서 sequence_length 가 20 이면 19 개의 예측 결과가 나오고, 각각의 값은 그 앞의 2개를 보고 예측한 결과임

class Conv1DModel( nn.Module ):
    def __init__(self, input_size, hidden_size ):
        super(Conv1DModel, self).__init__()

        self.conv1d = nn.Conv1d( in_channes=input_size, out_channels=hidden_size, kernel_size=2, stride=1 ) # padding 0 이라 h, w 가 1씩 줄어듦
        self.fc = nn.Linear( hidden_size, 1 )

    def forward(self, x):
        x = x.transpose( 1, 2 )     # [N, sequence_length, 1]       ->  [N, 1, sequence_length]
        x = self.conv1d( x )        # [N, 1, sequence_length]       ->  [N, 1, sequence_length - 1]     # padding: 0
        x = x.transpose( 1, 2 )     # [N, 1, sequence_length - 1]   ->  [N, sequence_length - 1, 1]
        return self.fc( x )

In [ ]:
### RNN 모델 구성 (학습 및 평가 함수는 동일)

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size ):
        super(RNNModel, self).__init__()

        self.rnn = nn.RNN( input_size, hidden_size, num_layers, batch_first=True )      # LSTM 과 동일
        self.fc = nn.Linear( hidden_size, 1 )

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out)

In [ ]:
### Encoder-Decoder (RNN-RNN) 모델 구성 및 학습 / 평가 함수
    - 입력도 sequence (encoder), 학습할 결과도 sequence (decoder) 로 구성
    - 학습 및 평가도 결과 sequence 를 비교해야 함

## 1. Data Processing : convert_data_into_tensors() 함수 부분이 변경됨

sequence_length = 50        # 입력
traget_len = 10             # 출력 (예측 길이)

def create_enc_dec_sequence( data ) :
    features, labels = [], []
    for i in range( len(data) - sequence_length - target_len ) :
        features.append( data[ i : i + sequence_length ] )
        labels.append( data[ i + sequence_length : i + sequence_length + target_len ] )

    features = np.array( features, dtype=np.float32 )
    labels = np.array( labels, dtype=np.float32 )

    features = torch.tensor( features, dtype=torch.float32 )
    labels = torch.tensor( labels, dtype=torch.float32 )

    return features, labels

X_train_, y_train_ = create_enc_dec_sequences( train_scaled )
X_test_, y_test_ = create_enc_dec_sequences( test_scaled )


## 2. 모델 class

class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):        # input_size : sequence_length
        super(EncoderRNN, self).__init__()
        
        self.rnn = nn.RNN( input_size, hidden_size, num_layers, batch_first=True )

    def forward(self, x):
        _, h = self.rnn(x)
        return h


class DecoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):        # input_size : sequence_length
        super(DecoderRNN, self).__init__()

        self.rnn = nn.RNN( input_size, hidden_size, num_layers, batch_first=True )
        self.fc = nn.Linear( hidden_size, input_size )

    def forward(self, x, h):
        out, h = self.rnn(x, h)
        out = self.fc(out)
        return out, h


class RNNRNN(nn.Moduel):
    def __init__(self, input_size, hidden_size, num_layers):
        super(RNNRNN, self).__init__()
    
        self.encoder = EncoderRNN( input_size, hidden_size, num_layers )
        self.decoder = DecoderRNN( input_size, hidden_size, num_layers )

    def forward(self, source, target_len):
        h = self.encoder(source)
        predictions = []
        input = source[:, -1, :].unsqueeze(1)
        for t in range(target_len):
            out, h = self.decoder(input, h)
            predictions.append(out.squeeze(1))
            input = out
        outputs = torch.stack(predictions, dim=1)
        return outputs

model_ = RNNRNN( input_size, hidden_size, num_layers ).to(device)


## Train/Test 함수에서는 model( x_input, target_len ) 으로 target_len 이 인자로 들어가는 것만 다름

## RNNRNN 의 input_size 는 1
    - 각 시점 (sequence 50개) 마다 'Open' 1개의 값만 가지고 있으므로 inpuse_size 는 1
